# Thuc nghiem toi thieu khoa luan — 9 run con lai

Doi `JOB` o Cell 2 roi Run All. **Moi lan chay DUNG mot job.**

| JOB | Noi dung | Config | ~gio |
|---|---|---|---|
| `p3_m3`    | P3-NoKD-More, MIMIC 3% — **chay lai** lam hang day du cua bang ablation | advanced | 2,7 |
| `fmi_m6`   | Forget-MI, MIMIC 6%  | baseline | 3,0 |
| `p3_m6`    | P3-NoKD-More, MIMIC 6%  | advanced | 2,7 |
| `fmi_m10`  | Forget-MI, MIMIC 10% | baseline | 3,0 |
| `p3_m10`   | P3-NoKD-More, MIMIC 10% | advanced | 2,7 |
| `fmi_iu`   | Forget-MI, IU 3%     | baseline_iu | 2,5 |
| `p3_iu`    | P3-NoKD-More, IU 3%  | loku_iu  | 2,2 |
| `abl_fila` | P3 w/o Fisher/FILA, MIMIC 3% | advanced | 2,6 |
| `abl_ihl`  | P3 w/o IHL, MIMIC 3%         | advanced | 2,7 |
| `abl_mumr` | P3 w/o MU/MR, MIMIC 3%       | advanced | 2,7 |
| `p3_iu_lr2e4` | **MOI** — P3 tren IU, lr 2e-4 + grad_clip 0 (khop MIMIC) | loku_iu + override | 3,0 |
| `p3_m3_lr1e4` | **MOI** — P3 tren MIMIC 3%, lr 1e-4 (do nhay lr) | advanced + override | 2,7 |
| `p3_f1` | **MOI** — tang luc quen: λ_IHL 3.0 / λ_CE 0.5 / FILA-anh 1.0 | advanced + override | 2,7 |
| `p3_f2` | **MOI** — λ_IHL 5.0 / λ_CE 0.25 / FILA-anh 1.0 | advanced + override | 2,7 |
| `p3_f3` | **MOI** — nhu f2 + `loku_subtract_scale` 1.5 | advanced + override | 2,7 |
| `p3_f4` | **MOI** — nhu f3 + nhanh anh 3 block + fc1 (**~2% tham so**) | advanced + override | 2,8 |

**MIMIC 3%**: ban so sanh FMI vs P3 DA XONG, giu nguyen ket qua cu, khong chay lai.
Rieng `p3_m3` phai chay lai vi run cu thuc hien TRUOC ban sua reset RNG, nen khong
dung lam hang doi chung cua bang ablation duoc (3 ablation se chay tren day RNG khac).
`fmi_m3` KHONG can chay lai — ban sua RNG chi dong vao P3.

Uoc tinh gio la SUY RA tu run 3% (P3 9606s = 2,67h; FMI 6260s + 1046s chan doan = 2,03h;
FMI o day cong them CE-selector nen ~3h). Khong phai so do that cho 6/10%/IU.

## Hai job lr moi them (chay sau cung, khong dung len so cu)

`p3_iu_lr2e4` — job `p3_iu` cu nap `config_loku_iu_kaggle.yaml` voi **lr 5e-4 va
grad_clip 1.0**, trong khi P3 tren MIMIC dung **lr 2e-4 va grad_clip 0.0**. Bao cao lai
dang khang dinh "dung nguyen cau hinh khoa tu MIMIC, khong tuning lai theo IU" -> khong
dung voi run that. Da doi chieu toan bo cac khoa con lai (lora_r 8 / lora_alpha 16 /
dropout 0.05 / weight_decay 0.01 / warmup 0.1 / batch 16 / fisher 256-16 / splits 4-10):
**giong het nhau**, chi lech dung hai khoa nay. Job moi dat lai ca hai -> khoi phuc dung
cau hinh khoa. Ket qua cu GIU LAI lam hang "P3 tren IU voi lr 5e-4".

`p3_m3_lr1e4` — lr cua hai phuong phap khac nhau theo thiet ke (Forget-MI 1e-5 full-FT,
P3 2e-4 LoRA) va **khong ben nao duoc sweep**. Job nay chay P3 3% o 1e-4 de tra loi cau
"P3 thang o 3% co phai nho lr may man khong". Ket qua giong -> khong nhay lr; khac ->
ghi vao Han che.

Ca hai job dung `JOB_OVR` o Cell 2, KHONG sua config file, nen moi job cu chay lai van
ra dung so cu.

## Bon run TANG LUC QUEN — `p3_f1` .. `p3_f4` (MIMIC 3%, 4 account song song)

**Van de.** P3 hien tai **quen qua it**: Df-AUC 0.683 trong khi gold la 0.498 (og 0.731).
Nguyen nhan KHONG phai "chi cap nhat 1,27% tham so" — bang chung: Dt-AUC 0.704 **cao hon
ca og** 0.695 va Test-CE 1.829 **thap hon** og 2.108, tuc mo hinh dang gioi len o tac vu
giu lai. Phia GIU LAI dang ap dao phia QUEN. Ti le Forget-CE/Test-CE = 1.13 -> con rat
nhieu du dia day manh ma chua cham nguong quen qua da.

**Cach day.** Chi qua `lambda_ihl` (IHL bi chan trong [0,2] nen an toan) va noi `lambda_ce`.
KHONG dung `lambda_uu`/`lambda_mu`: (1) chung la −Dist khong co day -> phan ky kieu
NegGrad+ (`forget_ce` 54 -> nan) va (2) chung lay cung tu `SCHEMES['uni']` trong
`forgetmi_p3_cand.py` nen override se bi **bo qua im lang**.

| | hien tai | f1 | f2 | f3 | f4 |
|---|---:|---:|---:|---:|---:|
| λ_UR / λ_UU / λ_MU / λ_MR | 1/3, 1/3, 1/6, 1/6 | (cung) | (cung) | (cung) | (cung) |
| λ_KD | 0 | 0 | 0 | 0 | 0 |
| **λ_CE** | 1.0 | 0.5 | 0.25 | 0.25 | 0.25 |
| **λ_IHL** | 1.0 | 3.0 | 5.0 | 5.0 | 5.0 |
| `loku_subtract_scale` | 1.0 | 1.0 | 1.0 | 1.5 | 1.5 |
| `loku_image_subtract_scale` | 0.5 | 1.0 | 1.0 | 1.0 | 1.0 |
| `lora_image_last_k_blocks` | 2 | 2 | 2 | 2 | 3 |
| `lora_image_include_fc1` | 0 | 0 | 0 | 0 | 1 |
| % tham so | 1,27% | 1,27% | 1,27% | 1,27% | ~2% |

**Tieu chi chon** (chot TRUOC khi chay): Df-AUC gan gold 0.498 nhat, **voi hai lan can**
`Dt-AUC >= 0.615` (duoi muc gold la pha mo hinh) va `Forget-CE/Test-CE <= 1.5` (tren
nguong nay la quen qua da -> MIA thap gia). Run nao pha lan can thi **loai**.

**Kiem ngay 5 phut dau run** (sai thi tat, dung de chay het 2,7h):
- `Override: lambda_ihl = ...` va `Override: lambda_ce = ...` phai xuat hien.
- `📊 Trainable:` phai la **1,451,008 (1.281%)** voi f1–f3; voi **f4 phai LON HON** — neu
  f4 van 1,451,008 thi `lora_image_include_fc1` chua vao.
- Dong dau vong train phai la `scheme=uni_nokd (base=uni)`.

Cell 4 tu dat `RUN_REF=False` cho ca 4 job (vi co `JOB_OVR`) -> khong eval lai OG/GOLD,
tiet kiem ~25 phut. So tham chieu da co: og 0.731 / gold 0.498.

## Chi so bao cao (theo danh sach thuc nghiem)
Chi dung **S2 (Closest CE)** + **E30**. S1/S3/S4 van duoc ghi ra file nhung KHONG bao cao.
Bang chinh chi lay: Df-AUC/F1, Dt-AUC/F1, MIA, Forget-CE, Test-CE, tham so, **T_core**, GPU peak.
KHONG dua T_selection / T_eval / pipeline vao bang chinh.


In [ ]:
# Cell 1: setup + CHOT CHAN code da push
import os, subprocess
WORK='/kaggle/working'; REPO=f'{WORK}/Forget-MI-LoKU'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','https://github.com/nhnhu146/Forget-MI-LoKU.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
os.chdir(REPO)
assert os.path.exists('training/adv_common.py'),'push code truoc + re-import notebook'
_adv=open('training/adv_common.py').read()
assert 'ce_selector' in _adv and 'checkpoint_selection_' in _adv, \
    '❌ adv_common CHUA co hook CE-selector -> chay `git push` code MOI roi moi Save Version!'
assert 'OnlineCESelector' in open('training/ce_selector_pilot.py').read(), '❌ git push code moi truoc!'
assert os.path.exists('training/forgetmi_p3_cand.py'), '❌ chua push forgetmi_p3_cand.py!'
print('✅ Code CE-selector da co (hook + OnlineCESelector).')
subprocess.run(['pip','install','-q','pydicom','scikit-image','scikit-learn','pyyaml','wandb','seaborn==0.13.2'],check=True)
subprocess.run(['pip','install','-q','transformers==4.38.0','peft==0.10.0','accelerate==0.27.0'],check=True)
import torch; assert torch.cuda.is_available(),'Bat GPU'
print('Commit:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())
print('GPU   :',torch.cuda.get_device_name(0))


In [ ]:
# Cell 2: CHON JOB + path discovery
import glob, os

# --- DOI DUNG 1 DONG NAY ---
JOB = 'p3_f1'          # p3_m3|fmi_m6|p3_m6|fmi_m10|p3_m10|fmi_iu|p3_iu|abl_fila|abl_ihl|abl_mumr
                       # hai job lr: p3_iu_lr2e4 | p3_m3_lr1e4
                       # 4 run TANG LUC QUEN (MIMIC 3%, chay 4 account song song):
                       #   acc1 = p3_f1   acc2 = p3_f2   acc3 = p3_f3   acc4 = p3_f4

SEED   = 42
EPOCHS = 30

# job -> (dataset, forget%, phuong phap, ablation)
JOBS = {
 # p3_m3: CHAY LAI P3-More 3% bang code da reset RNG -> lam hang "P3 day du" cua
 # bang ablation. Bang so sanh FMI vs P3 o MIMIC 3% VAN DUNG KET QUA CU.
 'p3_m3'   : ('mimic', 3,  'p3',  'none'),
 'fmi_m6'  : ('mimic', 6,  'fmi', 'none'),
 'p3_m6'   : ('mimic', 6,  'p3',  'none'),
 'fmi_m10' : ('mimic', 10, 'fmi', 'none'),
 'p3_m10'  : ('mimic', 10, 'p3',  'none'),
 'fmi_iu'  : ('iu',    3,  'fmi', 'none'),
 'p3_iu'   : ('iu',    3,  'p3',  'none'),
 'abl_fila': ('mimic', 3,  'p3',  'fisher_fila'),
 'abl_ihl' : ('mimic', 3,  'p3',  'ihl'),
 'abl_mumr': ('mimic', 3,  'p3',  'mu_mr'),
 # --- hai job lr, them sau khi 12 run chinh da xong ---
 'p3_iu_lr2e4': ('iu',    3,  'p3',  'none'),
 'p3_m3_lr1e4': ('mimic', 3,  'p3',  'none'),
 # --- 4 run TANG LUC QUEN, deu la MIMIC 3% + P3-NoKD-More, chi khac JOB_OVR ---
 'p3_f1'   : ('mimic', 3,  'p3',  'none'),
 'p3_f2'   : ('mimic', 3,  'p3',  'none'),
 'p3_f3'   : ('mimic', 3,  'p3',  'none'),
 'p3_f4'   : ('mimic', 3,  'p3',  'none'),
}

# Override RIENG cho tung job. Rong = giu nguyen config -> moi job cu chay lai van ra
# dung so cu. KHONG sua file config vi con phai tai lap 12 run da bao cao.
#   p3_iu_lr2e4: config_loku_iu dung lr 5e-4 + clip 1.0; MIMIC dung 2e-4 + clip 0.0.
#                Dat lai ca hai = khoi phuc dung cau hinh da khoa tu MIMIC.
#   p3_m3_lr1e4: chi doi lr (config advanced da co grad_clip 0.0 san).
#
# p3_f1..f4 = do duong cong danh doi QUEN <-> TIEN ICH. Ly do: P3 hien tai UNDER-FORGET
#   (Df-AUC 0.683 vs gold 0.498) va dang giu lai QUA manh (Dt-AUC 0.704 > og 0.695,
#   Test-CE 1.829 < og 2.108) -> phia giu lai ap dao phia quen.
#   Chi tang luc quen qua lambda_ihl (IHL bi chan trong [0,2], an toan) va noi lambda_ce.
#   KHONG dung lambda_uu/lambda_mu: chung la -Dist khong co day -> phan ky kieu NegGrad,
#   VA chung lay cung tu SCHEMES['uni'] nen override cung bi bo qua im lang.
#   f1->f4 tang dan; f4 la run DUY NHAT lam phinh % tham so (1.27% -> ~2%).
JOB_OVR = {
 'p3_iu_lr2e4': {'learning_rate': 2.0e-4, 'grad_clip': 0.0},
 'p3_m3_lr1e4': {'learning_rate': 1.0e-4},
 # lambda_ihl 1.0 -> 3.0 | lambda_ce 1.0 -> 0.5 | FILA nhanh anh 0.5 -> 1.0
 'p3_f1': {'lambda_ihl': 3.0, 'lambda_ce': 0.5,
           'loku_image_subtract_scale': 1.0},
 # day manh hon: IHL 5.0, CE 0.25
 'p3_f2': {'lambda_ihl': 5.0, 'lambda_ce': 0.25,
           'loku_image_subtract_scale': 1.0},
 # + tru FILA qua da o ca hai nhanh (xoa manh ngay tu khoi tao)
 'p3_f3': {'lambda_ihl': 5.0, 'lambda_ce': 0.25,
           'loku_image_subtract_scale': 1.0, 'loku_subtract_scale': 1.5},
 # + noi suc chua nhanh ANH (noi moi chi so quen duoc do) -> % tham so tang
 # 1 thay cho True: apply_overrides chi parse int/float, con lai giu NGUYEN CHUOI
 # -> 'False' cung se la truthy. Luon dung 1/0 cho co bool.
 'p3_f4': {'lambda_ihl': 5.0, 'lambda_ce': 0.25,
           'loku_image_subtract_scale': 1.0, 'loku_subtract_scale': 1.5,
           'lora_image_last_k_blocks': 3, 'lora_image_include_fc1': 1},
}
assert JOB in JOBS, f'JOB phai thuoc {sorted(JOBS)}'
DATASET, FORGET_PCT, KIND, ABLATE = JOBS[JOB]

def fd(*slugs):
    for s in slugs:
        if os.path.isdir(f'/kaggle/input/{s}'): return f'/kaggle/input/{s}'
        h=glob.glob(f'/kaggle/input/datasets/*/{s}')
        if h: return sorted(h)[0]
    return None
def bins(root): return sorted(glob.glob(os.path.join(root,'**','pytorch_model.bin'),recursive=True),key=len)
def first_existing(root, rels):
    for r in rels:
        p=os.path.join(root,r)
        if os.path.exists(p): return p
    return None

tag=f'{DATASET}{FORGET_PCT}per'
OUT=f'/kaggle/working/kltn_{JOB}_s{SEED}'
RESULTS=f'/kaggle/working/results_{JOB}.csv'

if DATASET=='mimic':
    CONFIG = 'config_baseline_kaggle.yaml' if KIND=='fmi' else 'config_advanced_kaggle.yaml'
    DATA=fd('forget-mi-data'); MOD=fd('forget-mi-models-full','forget-mi-models')
    assert DATA and MOD,'Add forget-mi-data + forget-mi-models-full'
    BASE=os.path.dirname([b for b in bins(MOD) if 'training_original_model' in b][0])
    gh=[b for b in bins(MOD) if f'model_retrained_{FORGET_PCT}per' in b]
    GOLD=os.path.dirname(gh[0]) if gh else BASE; HAS_GOLD=bool(gh)
    TEXT=os.path.join(DATA,'data','metadata'); IMG=os.path.join(DATA,'data','img_data')
    SPLIT='./data_splits/mimic-cxr-sub-img-edema-split-manualtest.csv'
    FORGET=f'./data_splits/forget_set_{FORGET_PCT}per.csv'
else:
    CONFIG = 'config_baseline_iu_kaggle.yaml' if KIND=='fmi' else 'config_loku_iu_kaggle.yaml'
    DATA=fd('forget-mi-data-iu'); MOD=fd('forget-mi-models-iu'); MODRE=fd('forget-mi-models-iu-re')
    RAD=fd('chest-xrays-indiana-university')
    assert DATA and MOD and RAD,'Add forget-mi-data-iu + forget-mi-models-iu + forget-mi-models-iu-re + chest-xrays-indiana-university'
    ogb=[b for b in bins(MOD) if 'model_og' in b.lower() or 'base_model' in b.lower()] or bins(MOD)
    BASE=os.path.dirname(ogb[0])
    reb=(bins(MODRE) if MODRE else []) or [b for b in bins(MOD) if 'retrain' in b.lower()]
    GOLD=os.path.dirname(reb[0]) if reb else BASE; HAS_GOLD=bool(reb)
    tsv=glob.glob(os.path.join(DATA,'**','all_data.tsv'),recursive=True) or glob.glob('/kaggle/input/**/all_data.tsv',recursive=True)
    TEXT=os.path.dirname(tsv[0]) if tsv else first_existing(DATA,['data/metadata','metadata'])
    IMG=first_existing(DATA,['data/img_data','img_data']) or (first_existing(RAD,['images/images_normalized','images']) if RAD else None) or RAD
    sp=glob.glob(os.path.join(DATA,'**','iu-split.csv'),recursive=True) or glob.glob('/kaggle/input/**/iu-split.csv',recursive=True) or glob.glob(os.path.join(DATA,'**','*iu*split*.csv'),recursive=True)
    fg=glob.glob(os.path.join(DATA,'**',f'forget_set_{FORGET_PCT}per_iu.csv'),recursive=True) or glob.glob(f'/kaggle/input/**/forget_set_{FORGET_PCT}per_iu.csv',recursive=True)
    assert sp and fg,'Khong thay iu-split / forget_set_iu'
    SPLIT=sp[0]; FORGET=fg[0]

for n,p in {'BASE':BASE,'TEXT':TEXT,'IMG':IMG,'SPLIT':SPLIT,'FORGET':FORGET}.items():
    assert p and os.path.exists(p),f'Missing {n}: {p}'

COMMON={'forget_set_path':FORGET,'base_model_path':BASE,'bert_pretrained_dir':BASE,
        'retrained_model_path':GOLD,'text_data_dir':TEXT,'img_data_dir':IMG,
        'data_split_path':SPLIT,'results_csv_path':RESULTS,'use_noise':1}
# P3-NoKD-More: cau hinh DA KHOA tu MIMIC 3%, khong tuning lai theo 6/10%/IU
MLP_TXT='attention.output.dense|intermediate.dense|output.dense'
MORE={'lora_extra_target_modules':MLP_TXT,'lora_image_last_k_blocks':2}

RID=f'{JOB}_s{SEED}'; OD=f'{OUT}/{RID}'
print('JOB',JOB,'| DATASET',DATASET,FORGET_PCT,'% | KIND',KIND,'| ABLATE',ABLATE)
print('config',CONFIG,'| GOLD',HAS_GOLD,'| run id',RID)
if JOB_OVR.get(JOB):
    print('OVERRIDE rieng cua job:',JOB_OVR[JOB],
          '  <- phai thay lai o log Cell 3 dang "Override: <key> = <value>"')
print('BASE',BASE); print('FORGET',FORGET)


In [ ]:
# Cell 3: CHAY (30 epoch). Ca hai deu bat CE-selector de co S2.
import os, subprocess, time
env={**os.environ,'PYTHONPATH':'.','WANDB_MODE':'disabled',
     'PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True'}

if KIND=='p3':
    ovr=dict(COMMON); ovr.update(MORE)
    ovr.update({'id':RID,'output_dir':OD,'unlearn_epochs':EPOCHS,
                'ce_selector':1,'s4_delta':0.15,
                'history_csv_path':f'/kaggle/working/perepoch_{RID}.csv'})
    ovr.update(JOB_OVR.get(JOB, {}))     # override rieng cua job (rong voi moi job cu)
    cmd=['python','training/forgetmi_p3_cand.py','--config',CONFIG,'--seed',str(SEED),
         '--scheme','uni_nokd','--ablate',ABLATE,'--fresh','--override',
         ','.join(f'{k}={v}' for k,v in ovr.items())]
else:
    ovr=dict(COMMON)
    ovr.update({'id':RID,'output_dir':OD,'unlearn_epochs':EPOCHS,
                'evaluate_last_and_best':1,
                'ce_selector_out':f'{OD}/checkpoint_selection_forgetmi',
                'history_csv_path':f'/kaggle/working/perepoch_{RID}.csv'})
    ovr.update(JOB_OVR.get(JOB, {}))     # override rieng cua job (rong voi moi job cu)
    cmd=['python','training/forgetmi_partial.py','--config',CONFIG,'--seed',str(SEED),
         '--fresh','--override',','.join(f'{k}={v}' for k,v in ovr.items())]

print('='*72+f'\n{RID}\n'+'='*72)
t0=time.time()
try:
    subprocess.run(cmd,env=env,check=True); print(f'OK {RID}  wall {(time.time()-t0)/3600:.2f}h')
except subprocess.CalledProcessError as e:
    print('FAIL',RID,'rc=',e.returncode)


In [ ]:
# Cell 4: eval OG + GOLD lam moc tham chieu cho MUC QUEN NAY
#   MIMIC 3%  -> False: da co san tu run truoc (results_final_mimic3per.csv).
#   6%/10%/IU -> True o CA HAI job cua muc do. Co y: RESULTS la file rieng cho tung
#     job (results_<JOB>.csv), nen chay o ca hai thi moi file tu chua du OG/GOLD ->
#     lap bang khong phai ghep cheo hai file. KHONG can sua tay thanh False.
import os, subprocess
# Job bien the lr KHONG eval lai OG/GOLD: hai mo hinh tham chieu khong hoc gi trong run
# nay nen khong phu thuoc lr, va data split van y het (cung seed) -> tai dung so da co.
RUN_REF = (FORGET_PCT!=3 or DATASET!='mimic') and not JOB_OVR.get(JOB)

P3CFG = 'config_advanced_kaggle.yaml' if DATASET=='mimic' else 'config_loku_iu_kaggle.yaml'
def evalref(label, mpath):
    ovr=dict(COMMON); ovr['output_dir']=f'{OUT}/_ref'; ovr['results_csv_path']=RESULTS
    env={**os.environ,'PYTHONPATH':'.','WANDB_MODE':'disabled'}
    cmd=['python','training/forgetmi_eval_only.py','--config',P3CFG,'--seed',str(SEED),
         '--label',label,'--model_type','pretrained','--model_path',mpath,
         '--method','reference','--override',','.join(f'{k}={v}' for k,v in ovr.items())]
    print('eval-ref',label)
    try: subprocess.run(cmd,env=env,check=True)
    except subprocess.CalledProcessError as e: print('FAIL',label,e.returncode)

if RUN_REF:
    evalref(f'og_{tag}',BASE)
    if HAS_GOLD: evalref(f're_{tag}',GOLD)
    else: print('(khong co GOLD cho',tag,')')
else:
    print('RUN_REF=False -> dung lai OG/GOLD da co tu run truoc '
          '(MIMIC 3% da san, hoac job bien the lr khong lam doi mo hinh tham chieu)')


In [ ]:
# Cell 5: S2 + E30 + T_core + GPU peak — DUNG cac so nay cho bang Chuong 4
import glob, json, os, pandas as pd
pd.set_option('display.width',220)

# --- S2 (gold-free) ---
for f in sorted(glob.glob(f'{OUT}/**/selected_checkpoints.json',recursive=True)):
    d=json.load(open(f,encoding='utf-8'))['results']
    s2=d.get('S2_closest_ce',{})
    print('S2 (Closest CE) ->', os.path.basename(os.path.dirname(f)))
    if s2.get('epoch') is None:
        print('   khong chon duoc:',s2.get('note'))
    else:
        print(f"   E{s2['epoch']}  Df-AUC {s2['Df_AUC']}  Df-F1 {s2['Df_F1']}  "
              f"Dt-AUC {s2['Dt_AUC']}  Dt-F1 {s2['Dt_F1']}  MIA {s2['MIA']}  "
              f"fce {s2['forget_ce']}  nmval_ce {s2['nm_val_ce']}")

# --- E30 + tham chieu ---
if os.path.exists(RESULTS):
    print('\n===== E30 (last) + OG/GOLD =====')
    dr=pd.read_csv(RESULTS)
    cols=[c for c in ['id','method','checkpoint_kind','checkpoint','selected_epoch',
                      'Forget_AUC','Forget_Macro_F1','Test_AUC','Test_Macro_F1',
                      'Df_AUC','Df_F1','Dt_AUC','Dt_F1','MIA','forget_ce','test_ce',
                      'trainable_params','trainable_ratio'] if c in dr.columns]
    print(dr[cols].to_string(index=False))

# --- T_core + GPU peak ---
print('\n===== T_core + GPU peak =====')
for f in sorted(glob.glob(f'{OUT}/**/timing_*.json',recursive=True)):
    d=json.load(open(f,encoding='utf-8'))
    print(f"{d.get('method'):18} T_fisher {d.get('fisher_seconds',0):7.1f}s  "
          f"T_fila {d.get('fila_seconds',0):6.1f}s  T_train {d.get('train_seconds',0):8.1f}s  "
          f"=> T_core {d.get('core_seconds',0):8.1f}s")
    print(f"{'':18} peak {d.get('core_peak_allocated_gb',0):.2f} GB alloc / "
          f"{d.get('core_peak_reserved_gb',0):.2f} GB reserved  |  "
          f"params {d.get('trainable_params',0):,} ({100*d.get('trainable_ratio',0):.2f}%)")
    print(f"{'':18} GPU {d.get('gpu_name')}  selector {d.get('selector')}")

print('\nTAI VE: timing_*.json + selected_checkpoints.json + results_*.csv + perepoch_*.csv')
